<a href="https://colab.research.google.com/github/Pigwen/hands-on-sft/blob/main/Chapter_4_Formatting_Your_Dataset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install bitsandbytes datasets trl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 17.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 532.9/532.9 kB 24.0 MB/s eta 0:00:00


In [2]:
import torch
from datasets import load_dataset, Dataset
from peft import prepare_model_for_kbit_training, get_peft_model, LoraConfig
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, AutoModelForCausalLM, AutoConfig, \
    DataCollatorForLanguageModeling, DataCollatorWithPadding, \
    DataCollatorWithFlattening, BitsAndBytesConfig
#from trl import setup_chat_format
from trl.data_utils import pack_dataset
#from trl.extras.dataset_formatting import FORMAT_MAPPING, conversations_formatting_function
#from compatibility_functions import DataCollatorForCompletionOnlyLM


# Pre-Reqs

In [3]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("facebook/opt-350m")
quote = 'A noble spirit embiggens the smallest man.'
print(tokenizer.tokenize(quote))
print(tokenizer.encode(quote, add_special_tokens=False))

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/685 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/644 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/441 [00:00<?, ?B/s]

['A', 'Ġnoble', 'Ġspirit', 'Ġemb', 'igg', 'ens', 'Ġthe', 'Ġsmallest', 'Ġman', '.']
[250, 25097, 4780, 18484, 11702, 1290, 5, 15654, 313, 4]


# The Road So Far

In [4]:
import torch
from peft import prepare_model_for_kbit_training, get_peft_model, LoraConfig
from transformers import AutoModelForCausalLM, BitsAndBytesConfig

supported = torch.cuda.is_bf16_supported
compute_dtype = torch.bfloat16 if supported else torch.float32

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=compute_dtype
)

model_q4 = AutoModelForCausalLM.from_pretrained(
    "facebook/opt-350m",
    device_map="cuda",
    dtype=compute_dtype,
    quantization_config=bnb_config
)

model_q4 = prepare_model_for_kbit_training(model_q4)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

peft_model = get_peft_model(model_q4, lora_config)

pytorch_model.bin:   0%|          | 0.00/663M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/662M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/137 [00:00<?, ?B/s]

# Apply Template

In [5]:
from transformers import AutoTokenizer

repo_id = "microsoft/phi-3-mini-4k-instruct"
tokenizer_phi = AutoTokenizer.from_pretrained(repo_id)
print(tokenizer_phi.chat_template)

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/306 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/599 [00:00<?, ?B/s]

{% for message in messages %}{% if message['role'] == 'system' %}{{'<|system|>
' + message['content'] + '<|end|>
'}}{% elif message['role'] == 'user' %}{{'<|user|>
' + message['content'] + '<|end|>
'}}{% elif message['role'] == 'assistant' %}{{'<|assistant|>
' + message['content'] + '<|end|>
'}}{% endif %}{% endfor %}{% if add_generation_prompt %}{{ '<|assistant|>
' }}{% else %}{{ eos_token }}{% endif %}


In [6]:
messages = [
    {'role': 'system', 'content': 'You are a helpful AI assistant.'},
    {'role': 'user',  'content': 'What is the capital of Argentina?'},
    {'role': 'assistant', 'content': 'Buenos Aires.'}
]

formatted = tokenizer_phi.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
print(formatted)

<|system|>
You are a helpful AI assistant.<|end|>
<|user|>
What is the capital of Argentina?<|end|>
<|assistant|>
Buenos Aires.<|end|>
<|endoftext|>


In [7]:
inference_input = tokenizer_phi.apply_chat_template(
    messages[:-1],
    tokenize=False,
    add_generation_prompt=True
)
print(inference_input)

<|system|>
You are a helpful AI assistant.<|end|>
<|user|>
What is the capital of Argentina?<|end|>
<|assistant|>



## Supported Format

In [8]:
from datasets import Dataset

conversation_ds = Dataset.from_list([{"messages": messages}])
conversation_ds.features

{'messages': List({'content': Value('string'), 'role': Value('string')})}

In [9]:
from trl.extras.dataset_formatting import FORMAT_MAPPING

FORMAT_MAPPING['chatml'] == conversation_ds.features["messages"]

True

In [10]:
FORMAT_MAPPING

{'chatml': List({'content': Value('string'), 'role': Value('string')}),
 'instruction': {'completion': Value('string'), 'prompt': Value('string')}}

## BYOFF (Bring Your Own Formatting Function)

In [11]:
batch_messages = [
    [{'role': 'user',
      'content': 'What is the capital of Argentina?'},
     {'role': 'assistant',
      'content': 'Buenos Aires.'}],
    [{'role': 'user',
      'content': 'What is the capital of the United States?'},
     {'role': 'assistant',
      'content': 'Washington D.C.'}]
]

In [12]:
def byo_formatting_func1(examples):
  messages = examples["messages"]
  output_texts = tokenizer_phi.apply_chat_template(
      messages,
      tokenize=False,
      add_generation_prompt=False
  )
  return output_texts

In [13]:
ds_msg = Dataset.from_dict({"messages": batch_messages})
ds_msg.map(lambda v: tokenizer_phi(byo_formatting_func1(v)), batched=True)

Map:   0%|          | 0/2 [00:00<?, ? examples/s]

Dataset({
    features: ['messages', 'input_ids', 'attention_mask'],
    num_rows: 2
})

In [14]:
batch_prompts_completions = {
    'prompt': ['What is the capital of Argentina?',
               'What is the capital of the United States?'],
    'completion': ['Buenos Aires.',
                    'Washington D.C.']
}

In [15]:
def byo_formatting_func2(examples):
  instruction_template = "### Question:"
  response_template = "### Answer:"
  text = f"{instruction_template} {examples["prompt"]}\n"
  text += f"{response_template} {examples["completion"]}"
  text += tokenizer_phi.eos_token
  return text

In [16]:
ds_prompt = Dataset.from_dict(batch_prompts_completions)
ds_prompt.map(lambda v: tokenizer_phi(byo_formatting_func2(v)))

Map:   0%|          | 0/2 [00:00<?, ? examples/s]

Dataset({
    features: ['prompt', 'completion', 'input_ids', 'attention_mask'],
    num_rows: 2
})

In [17]:
def byo_formatting_func3(examples):
  instruction_template = "### Question:"
  response_template = "### Answer:"

  output_texts = []
  for i in range(len(examples)):
    output_text = f"{instruction_template} {examples["prompt"][i]}\n"
    output_text += f"{response_template} {examples["completion"][i]}"
    output_text += tokenizer_phi.eos_token
    output_texts.append(output_text)
  return output_texts

ds_prompt.map(lambda v: tokenizer_phi(byo_formatting_func3(v)), batched=True)

Map:   0%|          | 0/2 [00:00<?, ? examples/s]

Dataset({
    features: ['prompt', 'completion', 'input_ids', 'attention_mask'],
    num_rows: 2
})

## BYOFD(Bring Your Own Formatted Data)

In [18]:
def byo_formatting_func(examples):
  messages = examples["messages"]
  output_texts = tokenizer_phi.apply_chat_template(
      messages,
      tokenize=False,
      add_generation_prompt=False
  )
  return {"text": output_texts}

In [19]:
formatted_ds = ds_msg.map(byo_formatting_func, batched=True)
formatted_ds["text"]

Map:   0%|          | 0/2 [00:00<?, ? examples/s]

Column(['<|user|>\nWhat is the capital of Argentina?<|end|>\n<|assistant|>\nBuenos Aires.<|end|>\n<|endoftext|>', '<|user|>\nWhat is the capital of the United States?<|end|>\n<|assistant|>\nWashington D.C.<|end|>\n<|endoftext|>'])

# The Tokenizer

In [20]:
from transformers import AutoTokenizer, AutoConfig

repo_id = "microsoft/phi-3-mini-4k-instruct"
tokenizer_phi = AutoTokenizer.from_pretrained(repo_id)
config_phi = AutoConfig.from_pretrained(repo_id, trust_remote_code=True)

config.json:   0%|          | 0.00/967 [00:00<?, ?B/s]

configuration_phi3.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/microsoft/phi-3-mini-4k-instruct:
- configuration_phi3.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


In [21]:
tokenizer_phi("Let's tokenize this sentence!")

{'input_ids': [2803, 29915, 29879, 5993, 675, 445, 10541, 29991], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1]}

## Vocabulary

In [22]:
len(tokenizer_phi), config_phi.vocab_size

(32011, 32064)

In [23]:
sorted(tokenizer_phi.vocab.items(), key=lambda t: -t[1])[:11]

[('<|user|>', 32010),
 ('<|placeholder6|>', 32009),
 ('<|placeholder5|>', 32008),
 ('<|end|>', 32007),
 ('<|system|>', 32006),
 ('<|placeholder4|>', 32005),
 ('<|placeholder3|>', 32004),
 ('<|placeholder2|>', 32003),
 ('<|placeholder1|>', 32002),
 ('<|assistant|>', 32001),
 ('<|endoftext|>', 32000)]

In [24]:
tokenizer_phi.eos_token, tokenizer_phi.eos_token_id

('<|endoftext|>', 32000)

## The Tokenizer 7

In [25]:
tokenizer_phi.all_special_tokens

['<s>', '<|endoftext|>', '<unk>']

In [26]:
tokenizer_phi.special_tokens_map

{'bos_token': '<s>',
 'eos_token': '<|endoftext|>',
 'unk_token': '<unk>',
 'pad_token': '<|endoftext|>'}

In [27]:
tokenizer_phi.add_special_tokens({
    'cls_token': '<cls>',
    'sep_token': '<sep>',
    'mask_token': '<mask>'
})
tokenizer_phi.special_tokens_map

{'bos_token': '<s>',
 'eos_token': '<|endoftext|>',
 'unk_token': '<unk>',
 'sep_token': '<sep>',
 'pad_token': '<|endoftext|>',
 'cls_token': '<cls>',
 'mask_token': '<mask>'}

In [28]:
sorted(tokenizer_phi.vocab.items(), key=lambda t: -t[1])[:14]

[('<mask>', 32013),
 ('<sep>', 32012),
 ('<cls>', 32011),
 ('<|user|>', 32010),
 ('<|placeholder6|>', 32009),
 ('<|placeholder5|>', 32008),
 ('<|end|>', 32007),
 ('<|system|>', 32006),
 ('<|placeholder4|>', 32005),
 ('<|placeholder3|>', 32004),
 ('<|placeholder2|>', 32003),
 ('<|placeholder1|>', 32002),
 ('<|assistant|>', 32001),
 ('<|endoftext|>', 32000)]

## The EOS Token

In [29]:
tokenizer_phi.pad_token = tokenizer_phi.unk_token
tokenizer_phi.pad_token_id = tokenizer_phi.unk_token_id

tokenizer_phi.all_special_tokens

['<s>', '<|endoftext|>', '<unk>', '<sep>', '<cls>', '<mask>']

In [30]:
tokenizer_phi.special_tokens_map

{'bos_token': '<s>',
 'eos_token': '<|endoftext|>',
 'unk_token': '<unk>',
 'sep_token': '<sep>',
 'pad_token': '<unk>',
 'cls_token': '<cls>',
 'mask_token': '<mask>'}

In [31]:
if getattr(model, "config", None) is not None:
  model.config.pad_token_id = tokenizer_phi.pad_token_id
if (getattr(model, "generation_config", None) is not None):
  model.config.pad_token_id = tokenizer_phi.pad_token_id

NameError: name 'model' is not defined

## The PAD Token

In [32]:
tokenizer_phi.pad_token, tokenizer_phi.padding_side

('<unk>', 'left')

# Data Collators